In [0]:
%sql
use catalog end_to_end_projects;
use schema sports_equipments_sales;

In [0]:
df_silver = spark.table("sports_equipment_sales_silver")
display(df_silver.limit(5))
print(f"Count of Records in Silver Delta Table: {df_silver.count()}")
df_silver.printSchema()

# Dimension Tables

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *

> Dim Customers

In [0]:
dim_customers = df_silver.select(col("customer_id"), col("customer_name"), col("customer_city"), col("customer_state"), col("last_updated"))\
    .withColumn("row_num", row_number().over(Window.partitionBy("customer_id").orderBy(col("last_updated").desc())))\
        .filter(col("row_num") == 1).drop("row_num", "last_updated")
display(dim_customers.limit(5))
print(f"Count of records in dim customers table: {dim_customers.count()}")
dim_customers.printSchema()

- Dim Customers Surrogate Key

In [0]:
dim_customers = dim_customers.withColumn("customer_sk", row_number().over(Window.orderBy("customer_id")))
display(dim_customers.limit(5))


In [0]:
print(f"Count of Total Records: {dim_customers.count()}")
print(f"Count of Dim Customer Surrogate Keys: {dim_customers.select("customer_sk").distinct().count()}")

### Save dim_customer as a Managed Delta Table

In [0]:
dim_customers.write.format("delta").mode("overwrite").saveAsTable("dim_customers")

In [0]:
dim_customers = spark.table("dim_customers")
display(dim_customers.limit(5))
print(f"Count of Records in Dim Customers Delta Table: {dim_customers.count()}")
dim_customers.printSchema()

- dim_products

In [0]:
dim_products = df_silver.select("product_id", "product_name", "product_category", "product_brand", "last_updated")\
    .withColumn("row_num", row_number().over(Window.partitionBy("product_id").orderBy(col("last_updated").desc())))\
        .filter(col("row_num") == 1).drop("row_num", "last_updated")\
            .withColumn("product_sk", row_number().over(Window.orderBy("product_id")))

display(dim_products.limit(5))
print(f"Count of records in dim products table: {dim_products.count()}")
print(f"Count of Unique Dim Product Surrogate Keys: {dim_products.select("product_sk").distinct().count()}")
dim_products.printSchema()

In [0]:
dim_products.write.format("delta").mode("overwrite").saveAsTable("dim_products")

In [0]:
dim_products = spark.table("dim_products")
display(dim_products.limit(5))
print(f"Count of Records in Products Delta Table: {dim_products.count()}")
dim_products.printSchema()

> dim_store

In [0]:
dim_store = df_silver.select("store_id", "store_name", "store_city", "store_region", col("last_updated"))\
    .withColumn("row_num", row_number().over(Window.partitionBy("store_id").orderBy(col("last_updated").desc())))\
        .filter(col("row_num") == 1).drop("row_num", "last_updated")\
            .withColumn("store_sk", row_number().over(Window.orderBy("store_id")))

display(dim_store.limit(5))
print(f"Count of Records in dim Store Table: {dim_store.count()}")
print(f"Count of Unique Records in Dim Store Table: {dim_store.select("store_id").distinct().count()}")
dim_store.printSchema()

In [0]:
dim_store.write.format("delta").mode("overwrite").saveAsTable("dim_store")

In [0]:
dim_store = spark.table("dim_store")
display(dim_store.limit(5))
print(f"Count of Records in Dim Store Delta Table: {dim_store.count()}")
dim_store.printSchema()

> dim_salesperson

In [0]:
dim_salesperson = df_silver.select("salesperson_id", "salesperson_name", "last_updated")\
    .withColumn("row_num", row_number().over(Window.partitionBy("salesperson_id").orderBy(col("last_updated").desc())))\
        .filter(col("row_num")==1).drop("row_num", "last_updated")\
            .withColumn("salesperson_sk", row_number().over(Window.orderBy("salesperson_id")))
display(dim_salesperson.limit(5))
print(f"Count of salesperson record: {dim_salesperson.count()}")
print(f"Count of Unique Salesperson surrogate keys: {dim_salesperson.select("salesperson_sk").distinct().count()}")
dim_salesperson.printSchema()

In [0]:
dim_salesperson.write.format("delta").mode("overwrite").saveAsTable("dim_salesperson")

In [0]:
dim_salesperson = spark.table("dim_salesperson")
display(dim_salesperson.limit(5))
print(f"Count of Records in Dim Salesperson Delta Table: {dim_salesperson.count()}")
dim_salesperson.printSchema()

# Fact Table Creation

In [0]:
fact_sales = spark.table("sports_equipment_sales_silver")
display(fact_sales.limit(5))
print(f"Count of Records: {fact_sales.count()}")


In [0]:
fact_sales = spark.table("sports_equipment_sales_silver")
dim_customers = spark.table("dim_customers")

fact_sales = fact_sales.join(dim_customers, on="customer_id", how="inner")\
    .join(dim_products, on = "product_id", how = "inner")\
        .join(dim_store, on = "store_id", how = "inner")\
            .join(dim_salesperson, on = "salesperson_id", how = "inner")

display(fact_sales.limit(5))
print(f"Count after Customer Join: {fact_sales.count()}")

In [0]:
print(f"Count after All Dimension JOINs: {fact_sales.count()}")

In [0]:
fact_sales = fact_sales.select("sale_id", "customer_sk", "product_sk", "store_sk", "salesperson_sk", "sale_date", "quantity", "unit_price", "discount_amount", "tax_amount", "final_amount", "payment_method", "sale_status", "last_updated")
display(fact_sales.limit(5))
print(f"Count of Records from Selected Fact Sales Columns: {fact_sales.count()}")
print(f"Unique Sale ID Count: {fact_sales.select("sale_id").distinct().count()}")

In [0]:
fact_sales.write.format("delta").mode("overwrite").saveAsTable("fact_sales")

In [0]:
fact_sales = spark.table("fact_sales")
display(fact_sales.limit(5))
print(f"Count of Records in Delta Fact Sales Table: {fact_sales.count()}") 